# Controlled Task 5 behavior submissions for 16 November

Local preparation only. v18 tests peer-relative change against frozen scored v17. v19 adds a bot-regularity rank to v18, isolating the secondary hypothesis. Both retain 6,531 Task 5 alerts, `1 = anomaly`, and exact v17 Tasks 1–4. They use only saved unlabeled models and dataset cards; no public score is read or optimized.

In [1]:
from pathlib import Path
import sys, json, hashlib, joblib
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "pipeline" / "save_model.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
SRC = ROOT / "src"
import numpy as np, pandas as pd
from scipy.stats import rankdata
from src.pipeline.random_seed import set_seed
SEED = set_seed()
BASE = SRC / "submission/kaggle_t12345_t3log_t5_rankblend25_v17.csv"
base = pd.read_csv(BASE, dtype=str)
base_card = json.loads(BASE.with_suffix(".json").read_text(encoding="utf-8"))
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
assert base_card["output_sha256"] == sha(BASE)
sample = pd.read_csv(SRC / "submission/sample_submission.csv", usecols=["id"], dtype=str)
assert base.columns.tolist() == ["id", "task1", "task2", "task3", "task4", "task5"]
assert len(base) == len(sample) == 25889 and base.id.tolist() == sample.id.tolist()
assert base.id.is_unique and base.task5.astype(int).sum() == 6531
print("Frozen v17:", sha(BASE), "alerts", int(base.task5.astype(int).sum()))

Frozen v17: d4b7c7b32c8c051a3d398819523ddfde4373699a7c2c2537b30ed912b519a9c8 alerts 6531


In [2]:
def read_test(name):
    card = json.loads((SRC / "task5/data/eda" / f"{name}.json").read_text(encoding="utf-8"))
    assert set(card["splits"]) == {"test"}
    frame = pd.read_parquet(SRC / card["splits"]["test"]["path"])
    assert frame.player_id.is_unique and "is_anomaly" not in frame
    return frame
old = read_test("eda_1_v2")
new = read_test("eda_behavior_views_v3")
assert old.player_id.equals(new.player_id) and len(new) == len(base)
def saved_model(name, data_name):
    card_path = SRC / "task5/save" / f"{name}.json"
    card = json.loads(card_path.read_text(encoding="utf-8"))
    assert card["data_name"] == data_name
    return joblib.load(SRC / card["model_path"]), card_path
ocsvm, ocsvm_card = saved_model("t5_r2_ocsvm_nu10_engineered_v2", "eda_1_v2")
gmm, gmm_card = saved_model("t5_r2_gmm_engineered_v2", "eda_1_v2")
peer, peer_card = saved_model("t5_peer_change_v3", "eda_behavior_views_v3")
bot, bot_card = saved_model("t5_bot_regular_v3", "eda_behavior_views_v3")
old_X, new_X = old.drop(columns="player_id"), new.drop(columns="player_id")
print("Saved feature sources and Test row order verified")

Saved feature sources and Test row order verified


In [3]:
def rank(values):
    values = np.asarray(values, dtype=float)
    assert len(values) == len(base) and np.isfinite(values).all()
    return rankdata(values, method="average") / len(values)
def select(score, k=6531):
    chosen = np.lexsort((np.arange(len(score)), -np.asarray(score)))[:k]
    pred = np.zeros(len(score), dtype=np.int8)
    pred[chosen] = 1
    return pred
v17_rank = .5 * rank(ocsvm.anomaly_score(old_X)) + .5 * rank(gmm.anomaly_score(old_X))
assert np.array_equal(select(v17_rank), base.task5.astype(np.int8).to_numpy())
h1_rank = .5 * v17_rank + .5 * rank(peer.anomaly_score(new_X))
h2_rank = .8 * h1_rank + .2 * rank(bot.anomaly_score(new_X))
h1, h2 = select(h1_rank), select(h2_rank)
assert h1.sum() == h2.sum() == 6531
for name, pred, reference in [("v18 versus v17", h1, base.task5.astype(int).to_numpy()),
                              ("v19 versus v18", h2, h1)]:
    print(name, "add", int(((pred==1)&(reference==0)).sum()),
          "remove", int(((pred==0)&(reference==1)).sum()))

v18 versus v17 add 1628 remove 1628
v19 versus v18 add 887 remove 887


In [4]:
specs = [
    ("kaggle_t12345_t5_peerchange_v18", h1, "H1: peer-relative within-account change",
     "0.5*v17_rank + 0.5*peer_rank", "kaggle_t12345_t3log_t5_rankblend25_v17.csv"),
    ("kaggle_t12345_t5_peerchange_bot_v19", h2, "H2: mechanical regularity adds to H1",
     "0.8*h1_rank + 0.2*bot_rank", "kaggle_t12345_t5_peerchange_v18.csv"),
]
for name, pred, hypothesis, formula, comparison in specs:
    csv = SRC / "submission" / f"{name}.csv"
    meta = csv.with_suffix(".json")
    assert csv.exists() == meta.exists(), f"Incomplete destination: {name}"
    output = base.copy()
    output["task5"] = pred.astype(str)
    assert output[["id","task1","task2","task3","task4"]].equals(base[["id","task1","task2","task3","task4"]])
    assert output.notna().all().all() and output.task5.astype(int).sum() == 6531
    source = base.task5.astype(int).to_numpy() if name.endswith("v18") else h1
    detail = {"name": name, "date_prepared": "2026-11-15", "planned_slot_date": "2026-11-16",
        "status": "prepared locally; not uploaded", "base_submission": BASE.name,
        "base_sha256": sha(BASE), "comparison_submission": comparison,
        "changed_task": "task5", "hypothesis": hypothesis, "score_formula": formula,
        "alert_budget": 6531, "alerts": int(pred.sum()),
        "swapped_in_vs_comparison": int(((pred==1)&(source==0)).sum()),
        "swapped_out_vs_comparison": int(((pred==0)&(source==1)).sum()),
        "task5_orientation": "1 = anomaly", "rows": len(output),
        "model_cards": [p.relative_to(SRC).as_posix() for p in [ocsvm_card,gmm_card,peer_card,bot_card]],
        "dataset_cards": ["task5/data/eda/eda_1_v2.json", "task5/data/eda/eda_behavior_views_v3.json"],
        "seed": SEED}
    if csv.exists():
        saved = pd.read_csv(csv,dtype=str)
        existing = json.loads(meta.read_text(encoding="utf-8"))
        assert saved.equals(output) and existing["output_sha256"] == sha(csv)
        assert all(existing[k] == v for k,v in detail.items())
    else:
        output.to_csv(csv,index=False)
        detail["output_sha256"] = sha(csv)
        meta.write_text(json.dumps(detail,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print("Verified",csv.name,"SHA-256",sha(csv))

Verified kaggle_t12345_t5_peerchange_v18.csv SHA-256 7b22f62486a13dfeed473cca78bc881d3b21d2a028e76312e519e2c50a29e5e9
Verified kaggle_t12345_t5_peerchange_bot_v19.csv SHA-256 c9793042d3942ad949d4a5be94c123fe0f50d61ed3ed4fd8af8a1e78e7b6ad97
